In [0]:
import os
import torch
import pandas as pd
import mlflow
import mlflow.pyfunc
from transformers import AutoTokenizer, AutoModel
from mlflow.models import infer_signature

In [0]:
# 1. Download and save the model locally
model_download_dir = "/Workspace/Users/abishek.offic@gmail.com/Transformer-Model-Deployment/databricks-sbert-mlflow/model_files"

token = AutoTokenizer.from_pretrained("sentence-transformers/paraphrase-MiniLM-L6-v2")
model = AutoModel.from_pretrained("sentence-transformers/paraphrase-MiniLM-L6-v2")

token.save_pretrained(model_download_dir)
model.save_pretrained(model_download_dir)

In [0]:
# 2. Define the Custom PyFunc Model
class SBertCustomModel(mlflow.pyfunc.PythonModel):

    def load_context(self, context):
        # Retrieve the actual folder path from artifacts
        model_dir = context.artifacts["model_dir"]
        self.token = AutoTokenizer.from_pretrained(model_dir)
        self.model = AutoModel.from_pretrained(model_dir)

    def predict(self, context, model_input: pd.DataFrame):
        text = model_input["text"].tolist()
        inputs = self.token(text, padding=True, return_tensors="pt")

        with torch.no_grad():
            output = self.model(**inputs)
        
        # Mean pooling extraction loop
        embds = []
        for i in output.last_hidden_state:
            embds.append(i.mean(axis=0).flatten().tolist())

        return pd.DataFrame({"prediction": embds})

In [0]:
# 3. Create Sample Examples to Automatically Infer the Perfect Signature
input_example = pd.DataFrame({"text": ["This is a sample text"]})

# Define a mock run of the model prediction logic to capture the output layout
sample_output = pd.DataFrame({"prediction": [[0.1] * 384]}) # Mock vector match layout

# Let MLflow calculate the exact required schema mapping
signature = infer_signature(input_example, sample_output)

In [0]:
import os
os.getcwd()

In [0]:
mlflow.set_experiment('/Workspace/Users/abishek.offic@gmail.com/Transformer-Model-Deployment/SBert_Model')


In [0]:
with mlflow.start_run(run_name="SBert_Test_Run"):
    mlflow.pyfunc.log_model(
        name="SBert_Model_1",
        python_model=SBertCustomModel(),
        artifacts={"model_dir": model_download_dir},
        input_example=input_example,
        signature=signature
    )
print("Model logged successfully with correct signature!")

In [0]:
model_name = "workspace.default.SBert_Model_1"
logged_model_uri = "models:/m-21b581c86ff1463cbb953036b9296711"

mlflow.register_model(logged_model_uri, model_name)